# Backtesting

In [1]:
import os
os.getcwd()
os.chdir('C:/Users/finle/Dev/Dissertation')

from src.backtests import *
import pandas as pd

spy = pd.read_csv('data/SPY_data.csv')
log_returns = spy['log_returns']

# Historical data

In [ ]:
historicals = pd.read_csv('data/historical_rolling_forecasts.csv')

print(historicals['historical_rolling_95'].first_valid_index())

historicals = historicals.iloc[50:]
log_returns = log_returns.iloc[50:]
print(log_returns)


In [13]:

def run_full_bactest(I: Series, alpha:float) -> dict:
    result  = {}
    
    k = kupiec_test(I, alpha)
    result.update(
        {'kupiec_test':k})
    c = christofferssen_test(I)
    result.update({'Christoffersen_independence': c})
    
    cc = christofferssen_conditional_test(k['LR_test'],c['Christofferssen LR test'])
    result.update({"Conditional_Coverage": cc})
    
    du = duration_test_unconditional(I, alpha)
    result.update({"Unconditional_Duration": du})
    
    dc = duration_test_conditional(I), 
    result.update({'Conditional_Duration': dc})
    return result
    

historical_95_I = backtest_var(log_returns, historicals['historical_rolling_95'])

historical_95_results = run_full_bactest(historical_95_I,0.05)
historical_95_results


{'kupiec_test': {'LR_test': np.float64(3.39487172569261),
  '1% significance:': np.float64(6.6348966010212145),
  '5% significance:': np.float64(3.841458820694124),
  'p-value': np.float64(0.06539944994813807)},
 'Christoffersen_independence': {'Christofferssen LR test': np.float64(21.037059884964947),
  '1% significance:': np.float64(6.6348966010212145),
  '5% significance:': np.float64(3.841458820694124),
  'p-value': np.float64(4.504849646425058e-06)},
 'Conditional_Coverage': {'LR_CC': np.float64(24.431931610657557),
  '1% critical': np.float64(9.21034037197618),
  '5% critical': np.float64(5.991464547107979),
  'p-value': np.float64(4.950778082601914e-06)},
 'Unconditional_Duration': {'LR_duration': np.float64(3.1503620004632467),
  'p-value': np.float64(0.0759101208016999),
  'lambda_hat': np.float64(0.05573440643863179)},
 'Conditional_Duration': ({'LR_weibull': np.float64(326.81123924515214),
   'p-value': np.float64(0.0),
   'k_hat': np.float64(0.6865727107045453)},)}

In [14]:

historical_99_I = backtest_var(log_returns, historicals['historical_rolling_99'])

historical_99_results = run_full_bactest(historical_99_I,0.01)
historical_99_results

{'kupiec_test': {'LR_test': np.float64(19.62676016656178),
  '1% significance:': np.float64(6.6348966010212145),
  '5% significance:': np.float64(3.841458820694124),
  'p-value': np.float64(9.414139167773783e-06)},
 'Christoffersen_independence': {'Christofferssen LR test': np.float64(11.981829219707947),
  '1% significance:': np.float64(6.6348966010212145),
  '5% significance:': np.float64(3.841458820694124),
  'p-value': np.float64(0.0005372182352259003)},
 'Conditional_Coverage': {'LR_CC': np.float64(31.608589386269728),
  '1% critical': np.float64(9.21034037197618),
  '5% critical': np.float64(5.991464547107979),
  'p-value': np.float64(1.368617297270447e-07)},
 'Unconditional_Duration': {'LR_duration': np.float64(18.759502158610985),
  'p-value': np.float64(1.4828267097422021e-05),
  'lambda_hat': np.float64(0.016757520694528568)},
 'Conditional_Duration': ({'LR_weibull': np.float64(247.12153079831296),
   'p-value': np.float64(0.0),
   'k_hat': np.float64(0.5415408741936569)},)}

In [ ]:
all_results = {
    'historical_95_results':historical_95_results,
    'historical_99_results': historical_99_I}



In [ ]:
def format_comparison_table(results:dict):
    def format_p(lr,p):
        if np.isnan(lr):
            return "-"
        stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p <0.05 else ""))
        return f"{lr:.3f}{stars}"
    
    rows = []
    for names, res in results.items():
        
        